In [1]:
import whisper

In [2]:
model = whisper.load_model("base")

In [3]:
result = model.transcribe("educational_video.mp4")

/Users/anikparui/Desktop/Project/RAG/rag_venv/lib/python3.13/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


In [4]:
transcript = result["text"]

In [5]:
print(transcript)

 Oh no! That was close! Thank you doctor! Oh no little kitty! Don't thank me! Instead thanks to my brain! Oh! Hello friends! Yes you heard it right! The reason we can do all these awesome things in our lives is due to an essential organ in our body called the brain. So today let us learn about this vital subject that helps us to learn about the vital subjects and explore the amazing world of the brain. Zoo-man! Your brain is basically the boss of your body as it controls everything you do. Things like learning, thinking, feeling, dancing, even breathing and your heart rate. It's due to your brain you can pull pranks on your siblings and friends and you won't believe but not even supercomputers can match its powerful ability to download, understand and react to the volume of information coming to you through your senses. So how does the brain manage all this? Different parts of your brain control different functions. So let us start with the larger spot called the cerebrum that takes up

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

/Users/anikparui/Desktop/Project/RAG/rag_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
#Split data
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 100)
docs = text_splitter.split_text(transcript)

In [8]:
print("The number of documents = ", len(docs))

The number of documents =  7


In [9]:
#loading huggingface embedding class
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings()

vector = embeddings.embed_query("hello world!")

/var/folders/49/dlfryms541x6nvjcz5yz0hh80000gn/T/ipykernel_28642/714104251.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/var/folders/49/dlfryms541x6nvjcz5yz0hh80000gn/T/ipykernel_28642/714104251.py:3: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
Loading weights: 100%|██████████████████████| 199/199 [00:00<00:00, 6874.89it/s]


In [10]:
print(len(vector))

768


In [11]:
#creating a vector database to store all the values of vector embeddings
# the vector database is named as vectorstore

from langchain_chroma import Chroma
vectorstore = Chroma.from_texts(texts=docs, embedding=embeddings)

In [12]:
# we want to use this vector database as retriver
# the task of retriver is to use those documents that matches the user's query

retriever = vectorstore.as_retriever(search_type = "similarity", search_kwargs = {"k" : 5})
retrieved_docs = retriever.invoke("brain")

In [13]:
# printing the length of the retrieved document

print("Length of the retrived document = ", len(retrieved_docs))

Length of the retrived document =  5


In [14]:
# print the content of any document

print(retrieved_docs[2].page_content)

the brain. Zoo-man! Your brain is basically the boss of your body as it controls everything you do. Things like learning, thinking, feeling, dancing, even breathing and your heart rate. It's due to your brain you can pull pranks on your siblings and friends and you won't believe but not even supercomputers can match its powerful ability to download, understand and react to the volume of information coming to you through your senses. So how does the brain manage all this? Different parts of your


In [16]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

# Use Groq API key to load the Groq llm
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key = api_key, model_name = "llama-3.3-70b-versatile")

prompt_template = """
You are a factual assistant.

Answer the question ONLY using the context below.
- Do NOT use prior knowledge
- Do NOT refuse unless context is empty
- If answer exists in context, give it directly
- Give full length answer

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=prompt_template
)

# Chain
llm_chain = prompt | llm | StrOutputParser()

In [17]:
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | llm_chain
)

In [18]:
#Input question
question = "What are the parts that a brain consist of?"

#answer
response = rag_chain.invoke(question)
print(response)

The parts of the brain mentioned in the context are: 
1. The cerebrum: It takes up to 85% of the brain, is the thinking part, and controls muscles, allowing activities like walking, dancing, and learning.
2. The spinal cord: It controls involuntary actions such as breathing, maintaining heart rate, and digesting food.
3. The amygdala: A small almond-shaped area responsible for emotions, survival instincts, and helping to store memories of events.
4. The cerebellum: It helps maintain balance and regulates motor movements.
5. The brain stem: Connected to the spinal cord, it controls involuntary actions like breathing and heart rate.
